In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 19. テキスト学習 — テキスト モデルテキスト テキスト [CPU/GPU ベンチマーク ⑧]

> **学習目標**
> - テキスト テキスト モデルテキスト(Causal LM)テキスト テキスト テキスト度テキスト
> - テキスト テキスト(teacher forcing)テキスト テキスト テキスト
> - 学習 データ テキスト 内容 テキスト テキスト
> - 学習ループ 速度テキスト CPU vs GPUテキスト 比較テキスト

## 19.1 テキスト学習テキスト テキスト — テキスト テキスト テキスト

LLM テキスト学習: テキスト テキスト テキスト テキスト テキスト.

$$\mathcal{L}_{\text{CLM}} = -\frac{1}{T} \sum_{t=1}^{T} \log P(w_t | w_{<t}; \theta)$$

- テキスト(causal): テキスト テキスト テキスト テキスト
- テキスト テキスト テキスト (Ch 05)
- テキスト テキスト度(self-supervised): テキスト = テキスト テキスト

## 19.2 テキスト テキスト (Teacher Forcing)

テキスト $\mathbf{x} = [w_0, w_1, \ldots, w_{T-1}]$テキスト テキスト:
- モデル テキスト: $[w_0, \ldots, w_{T-2}]$
- 答え: $[w_1, \ldots, w_{T-1}]$ (テキスト テキスト shift)

テキスト テキスト 順伝播テキスト $T-1$テキスト テキスト テキスト 学習 → テキスト.


In [ ]:

# テキスト モデル テキスト (Ch 18テキスト テキスト)
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def attention(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.W_qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask):
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2,
                 d_ff=256, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.token_emb(x) * (self.d_model ** 0.5) + self.pos_emb(positions)
        h = self.dropout(emb)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        for block in self.blocks:
            h = block(h, mask)
        h = self.ln_f(h)
        return self.lm_head(h)

print("MiniGPT Model テキスト テキスト")


In [ ]:
# Tiny Shakespeareテキスト テキスト学習
from llm_math.data import load_tiny_shakespeare
import numpy as np

text = load_tiny_shakespeare(max_chars=50000)
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
data = np.array([char_to_idx[c] for c in text])
print(f"テキスト Length: {len(text):,}, Vocabulary Size: {vocab_size}")

# テキスト テキスト
def get_batch(data, seq_len, batch_size):
    starts = np.random.randint(0, len(data) - seq_len - 1, batch_size)
    x = np.stack([data[s:s+seq_len] for s in starts])
    y = np.stack([data[s+1:s+1+seq_len] for s in starts])  # shift
    return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# 学習
model = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Parameter Count: {n_params:,}")

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

n_steps = 300
losses = []
import time
t0 = time.time()
for step in range(n_steps):
    x, y = get_batch(data, seq_len=64, batch_size=32)
    opt.zero_grad()
    logits = model(x)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    losses.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"Step {step+1}: loss={loss.item():.4f}")
print(f"\nTraining Time: {time.time() - t0:.1f}テキスト")


In [ ]:
# 学習 テキスト テキスト PPL
import matplotlib.pyplot as plt

ppl = np.exp(losses)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(losses, alpha=0.3, color='blue')
window = 20
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
axes[0].plot(range(window-1, len(losses)), smoothed, 'r-', linewidth=2)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Pretraining Loss Curve')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ppl, alpha=0.3, color='green')
axes[1].plot(range(window-1, len(ppl)), np.exp(smoothed), 'r-', linewidth=2)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('PPL')
axes[1].set_title(f'Perplexity (テキスト: {np.exp(losses[-1]):.2f})')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch19_pretraining.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# 学習 テキスト テキスト テキスト
def generate(model, prompt, n_new=200, temperature=0.8):
    model.eval()
    idx = [char_to_idx[c] for c in prompt]
    with torch.no_grad():
        for _ in range(n_new):
            x = torch.tensor([idx[-128:]], dtype=torch.long)  # テキスト 128
            logits = model(x)
            logit = logits[0, -1] / temperature
            probs = F.softmax(logit, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            idx.append(next_idx)
    return ''.join(idx_to_char[i] for i in idx)

print("=== Pretraining テキスト Generationテキスト テキスト ===\n")
print(generate(model, "ROMEO:\n", n_new=300, temperature=0.7))


## 19.3 [CPU/GPU ベンチマーク ⑧] 学習ループ 1 epoch 時間


In [ ]:
# 学習 速度 比較
import time
from llm_math.bench import time_fn

def train_one_step(model, x, y, opt, loss_fn):
    opt.zero_grad()
    logits = model(x)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    return loss

# CPU
model_cpu = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
opt_cpu = torch.optim.AdamW(model_cpu.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()
x_cpu, y_cpu = get_batch(data, 64, 32)
res_cpu = time_fn(train_one_step, model_cpu, x_cpu, y_cpu, opt_cpu, loss_fn, device='cpu', warmup=2, repeat=5)
print(f"CPU 1 step: {res_cpu['mean_ms']:.3f} ms ({res_cpu['std_ms']:.3f} std)")

if torch.cuda.is_available():
    model_gpu = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128).cuda()
    opt_gpu = torch.optim.AdamW(model_gpu.parameters(), lr=3e-4)
    x_gpu, y_gpu = x_cpu.cuda(), y_cpu.cuda()
    res_gpu = time_fn(train_one_step, model_gpu, x_gpu, y_gpu, opt_gpu, loss_fn, device='cuda', warmup=3, repeat=10)
    print(f"GPU 1 step: {res_gpu['mean_ms']:.3f} ms ({res_gpu['std_ms']:.3f} std)")
    print(f"Speed Improvement: {res_cpu['mean_ms'] / res_gpu['mean_ms']:.2f}x")
else:
    print("GPU テキスト. Modelテキスト テキスト GPU テキスト テキスト テキスト テキスト.")

# Mixed Precision Effect (GPUテキスト)
if torch.cuda.is_available():
    def train_step_amp(model, x, y, opt, loss_fn, scaler):
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x)
            loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        return loss
    scaler = torch.cuda.amp.GradScaler()
    res_amp = time_fn(train_step_amp, model_gpu, x_gpu, y_gpu, opt_gpu, loss_fn, scaler, device='cuda', warmup=3, repeat=10)
    print(f"GPU AMP 1 step: {res_amp['mean_ms']:.3f} ms (Mixed Precision)")


## 19.4 Gradient Accumulation

テキスト テキスト テキスト GPU テキスト テキスト テキスト: テキスト テキスト テキスト テキスト テキスト テキスト.

$$\theta \leftarrow \theta - \eta \frac{1}{K} \sum_{k=1}^{K} \nabla \mathcal{L}_k$$

テキスト テキスト テキスト = $K \times$ テキスト テキスト.


In [ ]:
# Gradient accumulation テキスト
def train_with_accumulation(model, data, opt, loss_fn, n_steps, accum_steps=4, batch_size=8):
    losses = []
    for step in range(n_steps):
        opt.zero_grad()
        total_loss = 0
        for _ in range(accum_steps):
            x, y = get_batch(data, 64, batch_size)
            logits = model(x)
            loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
            (loss / accum_steps).backward()  # テキスト
            total_loss += loss.item()
        opt.step()
        losses.append(total_loss / accum_steps)
    return losses

# 比較: batch_size=32 (1テキスト) vs batch_size=8 + accum=4 (テキスト テキスト テキスト)
torch.manual_seed(0)
model_a = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
opt_a = torch.optim.AdamW(model_a.parameters(), lr=3e-4)
torch.manual_seed(0)
model_b = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
opt_b = torch.optim.AdamW(model_b.parameters(), lr=3e-4)

print("Gradient accumulation vs テキスト (テキスト Effect Batch=32):")
# テキスト
losses_normal = []
for step in range(30):
    opt_a.zero_grad()
    x, y = get_batch(data, 64, 32)
    logits = model_a(x)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    opt_a.step()
    losses_normal.append(loss.item())

# accumulation
losses_accum = train_with_accumulation(model_b, data, opt_b, loss_fn, 30, accum_steps=4, batch_size=8)

plt.figure(figsize=(10, 4))
plt.plot(losses_normal, label='batch=32 (1テキスト)', alpha=0.7)
plt.plot(losses_accum, label='batch=8, accum=4 (テキスト)', alpha=0.7)
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Gradient Accumulation: テキスト Effect Batch Magnitude')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch19_grad_accum.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> テキスト Methodテキスト テキスト テキスト Effect. Memory テキスト テキスト accumulation テキスト.")


## 19.5 Mixed Precision (FP16/BF16)

- FP32 テキスト FP16/BF16テキスト 順伝播/バックプロパゲーション → テキスト テキスト, 速度 2テキスト
- テキスト FP32テキスト テキスト (テキスト テキスト)
- `torch.cuda.amp.autocast()`テキスト テキスト テキスト

CPUテキスト 高速化 テキスト, GPUテキスト テキスト テキスト.


## 19.6 要点

| テキスト | テキスト/テキスト | テキスト |
|---|---|---|
| Causal LM テキスト | $-\sum \log P(w_t\|w_{<t})$ | テキスト テキスト テキスト |
| テキスト テキスト | テキスト テキスト | テキスト テキスト Tテキスト 学習 |
| PPL | $\exp(\mathcal{L})$ | モデル テキスト度 |
| Gradient accumulation | $\frac{1}{K}\sum \nabla\mathcal{L}$ | テキスト テキスト |
| Mixed precision | FP16 順伝播, FP32 テキスト | 速度/テキスト |

## 演習問題

1. seq_len=32, 64, 128テキスト テキスト 学習 100テキスト テキスト PPLテキスト 比較テキスト.
2. batch_size=8, 16, 32, 64テキスト 学習テキスト テキスト 速度テキスト 比較テキスト.
3. Gradient accumulationテキスト batch_size=64テキスト テキスト テキスト (batch=16, accum=4).
4. Mixed precisionテキスト GPUテキスト テキスト 高速化テキスト テキスト.
5. 学習テキスト 1e-4, 3e-4, 1e-3テキスト 比較テキスト发散テキスト テキスト テキスト.

> 解答: `solutions/ch19_solutions.ipynb`
